# Geometry of Truth — GPT-2 + Sparse Autoencoder Baseline

This notebook starts **after EDA**.

## Goal
For each labeled true/false statement:

1. Tokenize with GPT-2 Small.
2. Run GPT-2 and capture the residual stream at `blocks.8.hook_resid_pre`.
3. Select the last non-padding token representation (768 dimensions).
4. Encode that representation with the pretrained `8-res-jb` SAE.
5. Save the resulting 24,576-dimensional sparse feature vector.
6. Compare SAE feature activation statistics between true and false statements.

> **Important:** Update the dataset-loading cell so `df` contains a text column and a binary truth label.


In [ ]:
# Optional: run once if these packages are not installed.
# %pip install -q transformer-lens sae-lens datasets pandas numpy matplotlib tqdm


In [ ]:
# 1. Imports

import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from transformer_lens import HookedTransformer
from sae_lens import SAE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 2. Load the Geometry of Truth dataframe

Your EDA is already complete, so this notebook only needs the processed dataframe.

After this cell, `df` must have:

- `statement`: the sentence given to GPT-2
- `label`: `1` for true and `0` for false

If your existing column names differ, rename them here.


In [ ]:
# CHANGE THIS CELL to point to the processed Geometry of Truth data from your EDA notebook.

# Example:
# df = pd.read_csv("../data/processed/geometry_of_truth.csv")

# If your columns have different names:
# df = df.rename(columns={"text": "statement", "truth": "label"})

# Required format:
# df = df[["statement", "label"]].dropna().reset_index(drop=True)

# Temporary validation guard:
required_columns = {"statement", "label"}
if "df" not in globals():
    raise NameError(
        "Load your processed Geometry of Truth dataframe into `df` first. "
        "It must contain columns named `statement` and `label`."
    )

missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[["statement", "label"]].dropna().reset_index(drop=True)
df["statement"] = df["statement"].astype(str)
df["label"] = df["label"].astype(int)

assert set(df["label"].unique()).issubset({0, 1}), "Labels must be 0/1."

print("Rows:", len(df))
print(df["label"].value_counts())
df.head()


## 3. Load GPT-2 Small

The pretrained SAE used in this project expects GPT-2 Small activations.

We will capture:

`blocks.8.hook_resid_pre`

For each token, this residual-stream activation has **768 values**.


In [ ]:
# 3. GPT-2 Small

model = HookedTransformer.from_pretrained("gpt2-small", device=device)
model.eval()

HOOK_NAME = "blocks.8.hook_resid_pre"

print("Model:", model.cfg.model_name)
print("d_model:", model.cfg.d_model)
print("Hook:", HOOK_NAME)


## 4. Load the pretrained SAE

This uses the same SAE family as the existing project baseline:

- SAE release: `gpt2-small-res-jb`
- SAE ID: `blocks.8.hook_resid_pre`

If your installed SAE-Lens version uses a different release string, change only `SAE_RELEASE`.


In [ ]:
# 4. Pretrained SAE

SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.8.hook_resid_pre"

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

sae.eval()

print("SAE input dimension:", sae.cfg.d_in)
print("SAE feature dimension:", sae.cfg.d_sae)

assert sae.cfg.d_in == model.cfg.d_model, (
    f"SAE expects {sae.cfg.d_in} dimensions but GPT-2 has {model.cfg.d_model}."
)


## 5. Sanity check with one statement

Before processing the whole dataset, verify the exact pipeline:

**statement → GPT-2 tokens → layer-8 residual stream → last-token 768-vector → SAE → sparse features**


In [ ]:
# 5. One-example sanity check

example_statement = df.iloc[0]["statement"]
example_label = int(df.iloc[0]["label"])

tokens = model.to_tokens(example_statement, prepend_bos=True)

with torch.inference_mode():
    _, cache = model.run_with_cache(
        tokens,
        names_filter=[HOOK_NAME],
    )

resid = cache[HOOK_NAME]          # [batch, sequence, 768]
last_resid = resid[:, -1, :]      # [batch, 768]
features = sae.encode(last_resid) # [batch, d_sae]

print("Statement:", example_statement)
print("Label:", example_label)
print("Tokens:", model.to_str_tokens(tokens))
print("Residual shape:", tuple(resid.shape))
print("Selected representation:", tuple(last_resid.shape))
print("SAE feature shape:", tuple(features.shape))
print("Active SAE features:", int((features[0] > 0).sum().item()))


## 6. Extract SAE features for the full dataset

We process statements in batches.

Because statements have different lengths, GPT-2 padding is used. The code uses the attention mask to select the **last real token**, not a padding token.


In [ ]:
# 6. Batched feature extraction

BATCH_SIZE = 16

# GPT-2 has no native pad token, so use EOS as padding.
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

def extract_sae_features(texts, batch_size=BATCH_SIZE):
    all_features = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Extracting SAE features"):
        batch_texts = texts[start:start + batch_size]

        encoded = model.tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=model.cfg.n_ctx - 1,
            add_special_tokens=False,
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        # Match the earlier project setup by prepending GPT-2's BOS/EOS token.
        bos = torch.full(
            (input_ids.shape[0], 1),
            model.tokenizer.bos_token_id,
            dtype=input_ids.dtype,
            device=device,
        )
        input_ids = torch.cat([bos, input_ids], dim=1)

        bos_mask = torch.ones(
            (attention_mask.shape[0], 1),
            dtype=attention_mask.dtype,
            device=device,
        )
        attention_mask = torch.cat([bos_mask, attention_mask], dim=1)

        with torch.inference_mode():
            _, cache = model.run_with_cache(
                input_ids,
                names_filter=[HOOK_NAME],
            )

            resid = cache[HOOK_NAME]  # [batch, seq, 768]

            # Number of real tokens per row minus one gives the last real token index.
            last_indices = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(resid.shape[0], device=device)

            last_resid = resid[batch_indices, last_indices, :]  # [batch, 768]
            batch_features = sae.encode(last_resid)             # [batch, d_sae]

        all_features.append(batch_features.detach().cpu())

        del cache, resid, last_resid, batch_features
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return torch.cat(all_features, dim=0)


texts = df["statement"].tolist()
sae_features = extract_sae_features(texts)

labels = torch.tensor(df["label"].values, dtype=torch.long)

print("Feature matrix:", tuple(sae_features.shape))
print("Labels:", tuple(labels.shape))

assert sae_features.shape[0] == len(df)


## 7. Separate true and false statements


In [ ]:
# 7. Split feature matrix by truth label

true_features = sae_features[labels == 1]
false_features = sae_features[labels == 0]

print("True feature matrix :", tuple(true_features.shape))
print("False feature matrix:", tuple(false_features.shape))


## 8. Baseline feature statistics

For every SAE feature we calculate:

- mean activation for true statements
- mean activation for false statements
- activation-frequency difference
- mean-activation difference

A positive difference means the feature is stronger/more common in the true group.  
A negative difference means it is stronger/more common in the false group.

These are **candidate associations**, not yet evidence that a feature causally represents truth.


In [ ]:
# 8. Feature statistics

true_mean = true_features.float().mean(dim=0)
false_mean = false_features.float().mean(dim=0)

true_frequency = (true_features > 0).float().mean(dim=0)
false_frequency = (false_features > 0).float().mean(dim=0)

mean_difference = true_mean - false_mean
frequency_difference = true_frequency - false_frequency

feature_stats = pd.DataFrame({
    "feature_id": np.arange(sae_features.shape[1]),
    "true_mean_activation": true_mean.numpy(),
    "false_mean_activation": false_mean.numpy(),
    "mean_activation_difference": mean_difference.numpy(),
    "true_activation_frequency": true_frequency.numpy(),
    "false_activation_frequency": false_frequency.numpy(),
    "activation_frequency_difference": frequency_difference.numpy(),
})

feature_stats.head()


## 9. Candidate features with the largest group differences


In [ ]:
# 9. Top candidate features

TOP_K = 20

top_true = (
    feature_stats
    .sort_values("mean_activation_difference", ascending=False)
    .head(TOP_K)
)

top_false = (
    feature_stats
    .sort_values("mean_activation_difference", ascending=True)
    .head(TOP_K)
)

print("Features with larger mean activation for TRUE statements:")
display(top_true)

print("\nFeatures with larger mean activation for FALSE statements:")
display(top_false)


## 10. Inspect which statements activate a candidate feature

This is useful for checking whether a high-ranking feature has a coherent interpretation rather than merely correlating with an accidental property of the dataset.


In [ ]:
# 10. Inspect a selected feature

def inspect_feature(feature_id, n=10):
    values = sae_features[:, feature_id].numpy()

    inspection = df[["statement", "label"]].copy()
    inspection["activation"] = values

    return inspection.sort_values("activation", ascending=False).head(n)


# Automatically inspect the feature with the largest TRUE-vs-FALSE mean difference.
candidate_feature = int(top_true.iloc[0]["feature_id"])

print("Candidate feature:", candidate_feature)
inspect_feature(candidate_feature, n=10)


## 11. Simple visualization

This plot is only a baseline visualization of the largest absolute mean-activation differences.


In [ ]:
# 11. Plot top absolute differences

plot_df = (
    feature_stats
    .assign(abs_difference=lambda x: x["mean_activation_difference"].abs())
    .nlargest(15, "abs_difference")
    .sort_values("mean_activation_difference")
)

plt.figure(figsize=(9, 6))
plt.barh(
    plot_df["feature_id"].astype(str),
    plot_df["mean_activation_difference"],
)
plt.axvline(0, linewidth=1)
plt.xlabel("Mean activation difference (True - False)")
plt.ylabel("SAE Feature ID")
plt.title("Geometry of Truth: Largest SAE Feature Differences")
plt.tight_layout()
plt.show()


## 12. Save baseline outputs

The saved feature matrix lets later notebooks run statistical tests, probes, feature inspection, steering, and faithfulness metrics **without rerunning GPT-2 + SAE extraction**.


In [ ]:
# 12. Save results

from pathlib import Path

OUTPUT_DIR = Path("../results/geometry_of_truth")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_stats.to_csv(
    OUTPUT_DIR / "geometry_of_truth_feature_stats.csv",
    index=False,
)

torch.save(
    {
        "features": sae_features,
        "labels": labels,
        "statements": df["statement"].tolist(),
        "hook_name": HOOK_NAME,
        "sae_release": SAE_RELEASE,
        "sae_id": SAE_ID,
    },
    OUTPUT_DIR / "geometry_of_truth_sae_features.pt",
)

print("Saved to:", OUTPUT_DIR.resolve())


# Baseline complete

At this point we have:

- GPT-2 Small layer-8 residual activations
- SAE representations for every Geometry of Truth statement
- true/false feature statistics
- candidate SAE features for further investigation
- reusable saved activations

## Next experiment

The next notebook should move beyond simple mean differences and test whether truth labels are **systematically decodable** from:

1. the original 768-dimensional GPT-2 residual representation, and
2. the 24,576-dimensional SAE representation.

That gives us a much stronger baseline for evaluating whether the SAE preserves truth-related information.
